<center>
    <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/Logos/organization_logo/organization_logo.png" width="300" alt="cognitiveclass.ai logo">
</center>


#### Import the required libraries we need for the lab.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

#### Read the dataset in the csv file from the URL


In [ ]:
boston_df=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv')

## Task 1: Become familiar with the dataset


**Question 1:** Display the first five rows of the DataFrame.


In [ ]:
boston_df.head()

**Question 2:** Check the data type of each column.


In [ ]:
boston_df.dtypes

**Question 3:** Count the missing values in each column.


In [ ]:
boston_df.isnull().sum()

**Question 4:** Identify duplicate entries.


In [ ]:
print('Number of duplicate rows:', boston_df.duplicated().sum())

**Question 5:** Determine the correlation matrix of the DataFrame.


In [ ]:
boston_df.corr()

## Task 2: Generate descriptive statistics and visualizations


**Question 6:** Descriptive statistics of the dataset.


In [ ]:
boston_df.describe()

**Scatter plot of NOX vs. INDUS**


In [ ]:
pyplot.figure(figsize=(8, 6))
sns.scatterplot(x='INDUS', y='NOX', data=boston_df)
pyplot.title('Scatter plot of NOX vs. INDUS')
pyplot.xlabel('Proportion of non-retail business acres per town (INDUS)')
pyplot.ylabel('Nitric oxide concentration (NOX)')
pyplot.show()

**Observation:** NOX and INDUS show a positive relationship: towns with a higher proportion of non-retail business acres tend to have higher nitric oxide concentrations.


**Histogram of pupil-to-teacher ratio (PTRATIO)**


In [ ]:
pyplot.figure(figsize=(8, 6))
sns.histplot(boston_df['PTRATIO'], kde=True)
pyplot.title('Distribution of Pupil to teacher ratio')
pyplot.xlabel('Pupil to teacher ratio')
pyplot.ylabel('Frequency')
pyplot.grid(True)
pyplot.show()

**Boxplot of the median value of owner-occupied homes (MEDV)**


In [ ]:
pyplot.figure(figsize=(8, 6))
sns.boxplot(y='MEDV', data=boston_df)
pyplot.title('Boxplot of Median value of owner-occupied homes')
pyplot.xlabel('Median value of owner-occupied homes (in $1000s)')
pyplot.ylabel('MEDV')
pyplot.show()

**Boxplot of MEDV vs. AGE group**

Create a new column that splits AGE into three groups.


In [ ]:
boston_df['age_group'] = pd.cut(
    boston_df['AGE'],
    bins=[0, 35, 70, 100],
    labels=['35 years and younger', 'Between 35 and 70 years', '70 years and older'],
    right=True
)

pyplot.figure(figsize=(10, 6))
sns.boxplot(x='age_group', y='MEDV', data=boston_df, hue='age_group', legend=False)
pyplot.title('Boxplot of MEDV vs. AGE group')
pyplot.xlabel('Age group of homeowners')
pyplot.ylabel('Median value of owner-occupied homes (in $1000s)')
pyplot.show()

**Bar plot of the Charles River variable (CHAS)**


In [ ]:
pyplot.figure(figsize=(6, 5))
sns.countplot(x='CHAS', data=boston_df)
pyplot.title('Bar Plot of Charles River Variable (CHAS)')
pyplot.xlabel('Bounded by River (1=Yes, 0=No)')
pyplot.ylabel('Count')
pyplot.show()

## Task 3: Use the appropriate tests to answer the questions provided


**Question 12:** Is there a significant difference in the variance of MEDV between homes bounded by the Charles River (CHAS = 1) and those that are not (CHAS = 0)? (Levene's test)

- H₀: The variances of MEDV are equal for both groups
- H₁: The variances of MEDV are not equal for both groups


In [ ]:
medv_river = boston_df[boston_df['CHAS'] == 1]['MEDV']
medv_no_river = boston_df[boston_df['CHAS'] == 0]['MEDV']

levene_stat, levene_p = scipy.stats.levene(medv_river, medv_no_river, center='mean')
print('Levene statistic:', levene_stat)
print('p-value:', levene_p)

if levene_p < 0.05:
    print('Reject H0: the variances are NOT equal.')
else:
    print('Fail to reject H0: the variances are equal.')

**Conclusion (Question 12):** Levene's test gives a statistic of 8.75 and a p-value of about 0.0032. Since the p-value is less than 0.05, we reject H₀. The variances of MEDV for homes bounded by the Charles River and those not bounded by it are significantly different.


**Question 13:** Is there a significant difference in the mean MEDV between homes bounded by the Charles River and those that are not? (T-test)

- H₀: The means of MEDV are equal for both groups
- H₁: The means of MEDV are not equal for both groups


In [ ]:
# Use equal_var based on the result of Levene's test
equal_var = levene_p >= 0.05
t_stat, t_p = scipy.stats.ttest_ind(medv_river, medv_no_river, equal_var=equal_var)
print('T-statistic:', t_stat)
print('p-value:', t_p)

if t_p < 0.05:
    print('Reject H0: the mean MEDV differs between the two groups.')
else:
    print('Fail to reject H0: no significant difference in mean MEDV.')

**Conclusion (Question 13):** Because Levene's test showed unequal variances, Welch's t-test (equal_var=False) was used. It gives t ≈ 3.11 and a p-value of about 0.0036. Since the p-value is less than 0.05, we reject H₀. There is a statistically significant difference in the mean median home value between homes bounded by the Charles River and those that are not (homes bounded by the river have a higher mean MEDV).


**Question 14:** Is there a difference in the average MEDV across the three age groups? (ANOVA)

- H₀: The average MEDV is the same for all three age groups
- H₁: At least one age group has a different average MEDV


In [ ]:
group_young = boston_df[boston_df['age_group'] == '35 years and younger']['MEDV']
group_middle = boston_df[boston_df['age_group'] == 'Between 35 and 70 years']['MEDV']
group_old = boston_df[boston_df['age_group'] == '70 years and older']['MEDV']

f_stat, f_p = scipy.stats.f_oneway(group_young, group_middle, group_old)
print('F-statistic:', f_stat)
print('p-value:', f_p)

if f_p < 0.05:
    print('Reject H0: at least one age group has a different mean MEDV.')
else:
    print('Fail to reject H0: the mean MEDV is the same across age groups.')

**Conclusion (Question 14):** The one-way ANOVA gives F ≈ 36.41 and a p-value of about 1.7e-15. Since the p-value is far below 0.05, we reject H₀. At least one age group has a different mean median home value. The boxplot suggests that homes with older owners (70 years and older) have the lowest values.


**Question 15:** Is there a relationship between nitric oxide concentration (NOX) and the proportion of non-retail business acres per town (INDUS)? (Pearson correlation)

- H₀: There is no linear relationship between NOX and INDUS
- H₁: There is a linear relationship between NOX and INDUS


In [ ]:
r, pearson_p = scipy.stats.pearsonr(boston_df['NOX'], boston_df['INDUS'])
print('Pearson correlation coefficient (r):', r)
print('p-value:', pearson_p)

if pearson_p < 0.05:
    print('Reject H0: NOX and INDUS are significantly linearly correlated.')
else:
    print('Fail to reject H0: no significant linear correlation.')

**Conclusion (Question 15):** The Pearson correlation coefficient is r ≈ 0.764 with a p-value of about 7.9e-98. Since the p-value is below 0.05, the correlation is statistically significant. NOX and INDUS have a strong positive linear relationship: towns with a larger proportion of non-retail business acres tend to have higher nitric oxide concentrations.


**Question 16:** What is the impact of an additional weighted distance to the five Boston employment centres (DIS) on the median value of owner-occupied homes (MEDV)? (Simple linear regression)

- H₀: The slope coefficient of DIS is equal to zero
- H₁: The slope coefficient of DIS is not equal to zero


In [ ]:
X = sm.add_constant(boston_df['DIS'])
y = boston_df['MEDV']
model = sm.OLS(y, X).fit()
print(model.summary())

**Conclusion (Question 16):** The slope coefficient for DIS is about 1.09 (intercept ≈ 18.39) with a p-value of about 1.2e-08. Since the p-value is below 0.05, we reject H₀ that the slope is zero. Each additional unit of weighted distance to the employment centres is associated with an increase of about $1,092 in the median home value. However, R-squared is only about 0.062, so DIS explains just 6.2% of the variation in MEDV.
